# EXAONE-Deep-7.8B QLoRA 파인튜닝

이 노트북은 전처리된 민원 데이터를 사용하여 EXAONE 모델을 QLoRA로 학습합니다.
런타임 유형이 **GPU (L4 또는 A100 권장)**로 설정되어 있는지 확인하세요.

In [ ]:
# 1. 필수 라이브러리 설치
!pip install -q -U transformers==5.3.0 datasets accelerate peft bitsandbytes trl==0.12.0 wandb python-dotenv

In [ ]:
# 2. EXAONE 모델 호환성을 위한 transformers 패치
import os
patch_path = "/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py"
patch_code = "\ndef check_model_inputs(func):\n    return func\n"

if os.path.exists(patch_path):
    with open(patch_path, "r") as f:
        content = f.read()
    
    if "def check_model_inputs" not in content:
        with open(patch_path, "a") as f:
            f.write(patch_code)
        print("Transformers patched successfully.")
    else:
        print("Transformers already patched.")

In [ ]:
# 3. 학습 스크립트 실행 환경 설정
import sys
import os

# 프로젝트 루트 경로를 PYTHONPATH에 추가
project_root = "/content/ondevice-ai-civil-complaint"
sys.path.append(project_root)

# W&B 오프라인 모드 설정 (학습 로그를 로컬에 저장)
os.environ["WANDB_MODE"] = "offline"

In [ ]:
# 4. QLoRA 파인튜닝 시작
TRAIN_PATH = "/content/ondevice-ai-civil-complaint/data/processed/civil_complaint_train.jsonl"
VAL_PATH = "/content/ondevice-ai-civil-complaint/data/processed/civil_complaint_val.jsonl"
PEFT_CONFIG_PATH = "/content/ondevice-ai-civil-complaint/src/training/peft_config.json"
OUTPUT_DIR = "/content/models/checkpoints/exaone-civil-qlora"
SCRIPT_PATH = "/content/ondevice-ai-civil-complaint/src/training/train_qlora.py"

!python {SCRIPT_PATH} \
    --train_path {TRAIN_PATH} \
    --val_path {VAL_PATH} \
    --peft_config_path {PEFT_CONFIG_PATH} \
    --output_dir {OUTPUT_DIR} \
    --epochs 1 \
    --batch_size 2 \
    --grad_accum 8 \
    --lr 2e-4 \
    --max_seq_length 2048